# Milestone 2 — Hierarchical parsing and standardization (Colab / local runner)

This notebook runs the `src/m2` parser end to end and measures it on the official testbed
(**Google Colab, CPU runtime**). It works locally as well: outside Colab the Drive and clone steps
are skipped and the repository's own `data/` folder is used.

Steps: configuration → environment → Drive → pack `data/` as a tar on Drive once and extract it to
local disk (timed separately) → smoke test on 20 papers → full run with resume → optional AnyStyle
refinement (off by default) → compaction → validation → statistics exports → measurements.

The parser never reads `references.json` (Milestone 3 must stay leakage free) and makes no network
calls; the tar step excludes `references.json` anyway.

In [ ]:
# ---- Configuration -------------------------------------------------------------------
REPO_URL     = "https://github.com/<group>/Intro2DS-Group3-Source.git"   # set to the group repo
BRANCH       = "ms2-parser"
DRIVE_DATA   = "/content/drive/MyDrive/Intro2DS-Group3/data"     # M1 corpus on Drive
DRIVE_TAR    = "/content/drive/MyDrive/Intro2DS-Group3/m2_data.tar"  # packed once, reused
DRIVE_OUT    = "/content/drive/MyDrive/Intro2DS-Group3/m2_output" # results copied back here
WORKERS      = "auto"     # 'auto' = min(CPUs, free RAM / per-worker estimate)
SMOKE_PAPERS = 20
RUN_FULL     = True       # False: smoke test only
LIMIT        = None       # e.g. 500 for a partial benchmark
RUN_ANYSTYLE = False      # optional bibitem refinement (needs Ruby + anystyle-cli)
COPY_BACK    = True
N_SHARDS     = 8

In [ ]:
import os, sys, time, json, shutil, subprocess, platform
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    REPO = Path("/content/Intro2DS-Group3-Source")
    DATA = Path("/content/data")
    OUT, WORK = Path("/content/m2_out/tables"), Path("/content/m2_out/m2")
    SMOKE = Path("/content/m2_smoke")
else:  # local: run from src/ or the repository root
    REPO = Path.cwd() if (Path.cwd() / "src" / "m2").is_dir() else Path.cwd().parent
    DATA = REPO / "data"
    OUT = Path(os.environ.get("M2_NB_OUT", str(REPO / "data" / "m2" / "nb" / "tables")))
    WORK = Path(os.environ.get("M2_NB_WORK", str(REPO / "data" / "m2" / "nb" / "m2")))
    SMOKE = Path(os.environ.get("M2_NB_SMOKE", str(REPO / "data" / "m2" / "nb" / "smoke")))
    RUN_FULL = os.environ.get("M2_NB_FULL", "0") == "1"
    LIMIT = int(os.environ.get("M2_NB_LIMIT", "40"))

def sh(*args, check=True):
    """Run a command with the notebook's interpreter and stream its output."""
    cmd = [str(a) for a in args]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode})")
    return p.returncode

print("Colab:", IN_COLAB, "| Python", platform.python_version(), "| repo:", REPO)

In [ ]:
# ---- Environment ---------------------------------------------------------------------
if IN_COLAB and not REPO.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, str(REPO)], check=True)
if IN_COLAB:
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt")
import psutil
print("CPUs:", os.cpu_count(), "| RAM total %.1f GB, available %.1f GB" % (
    psutil.virtual_memory().total / 2**30, psutil.virtual_memory().available / 2**30))
print("disk free: %.1f GB" % (shutil.disk_usage(str(REPO)).free / 2**30))

In [ ]:
# ---- Google Drive ---------------------------------------------------------------------
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# ---- Pack data/ once on Drive, extract to local disk (timed separately) -----------------
# Parsing tens of thousands of folders directly from the Drive mount is very slow.
timing = {}
if IN_COLAB:
    t0 = time.perf_counter()
    if not os.path.exists(DRIVE_TAR):
        src = Path(DRIVE_DATA)
        subprocess.run(["tar", "-cf", DRIVE_TAR, "--exclude=*/references.json",
                        "--exclude=tables", "--exclude=m2", "--exclude=run_logs",
                        "--exclude=report_exports", "-C", str(src.parent), src.name], check=True)
    timing["tar_create_s"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    if not DATA.exists():
        subprocess.run(["tar", "-xf", DRIVE_TAR, "-C", "/content"], check=True)
    timing["tar_extract_s"] = time.perf_counter() - t0
    print({k: round(v, 1) for k, v in timing.items()})
n_papers = sum(1 for p in DATA.iterdir() if p.is_dir() and not p.name.startswith(".")
               and p.name not in ("tables", "m2", "m2_nb", "run_logs", "report_exports"))
print("paper folders:", n_papers)

In [ ]:
# ---- Smoke test: 20 papers, 2 workers -----------------------------------------------------
shutil.rmtree(SMOKE, ignore_errors=True)
sh(sys.executable, "-m", "src.m2.run", "--data-dir", DATA, "--out", SMOKE / "tables",
   "--work-dir", SMOKE / "m2", "--workers", "2", "--limit", SMOKE_PAPERS, "--no-progress")
sh(sys.executable, "-m", "src.m2.validate", "--out", SMOKE / "tables", "--work-dir", SMOKE / "m2",
   "--data-dir", DATA)

In [ ]:
# ---- Full run (resumable: re-running this cell skips papers whose signature is unchanged) ---
args = [sys.executable, "-m", "src.m2.run", "--data-dir", DATA, "--out", OUT, "--work-dir", WORK,
        "--workers", WORKERS, "--resume"]
if not RUN_FULL:
    args += ["--limit", LIMIT or SMOKE_PAPERS]
elif LIMIT:
    args += ["--limit", LIMIT]
if RUN_ANYSTYLE:
    args += ["--refine-bibitem"]
t0 = time.perf_counter()
sh(*args)
timing["parse_wall_s"] = time.perf_counter() - t0

In [ ]:
# ---- Optional: AnyStyle refinement of low-confidence bibitems (off by default) -------------
# Needs Ruby. Accepted fields must appear in the original bibitem text; the parser works without it.
if RUN_ANYSTYLE and IN_COLAB:
    subprocess.run("apt-get -qq install -y ruby-full > /dev/null && gem install -q anystyle-cli",
                   shell=True, check=False)
    sh(sys.executable, "-m", "src.m2.tests.eval_gold", "--only", "bibitems", "--refine")

In [ ]:
# ---- Compaction, validation, statistics ----------------------------------------------------
t0 = time.perf_counter()
sh(sys.executable, "-m", "src.m2.run", "--out", OUT, "--work-dir", WORK, "--compact-only",
   "--n-shards", N_SHARDS)
timing["compact_s"] = time.perf_counter() - t0
rc = sh(sys.executable, "-m", "src.m2.validate", "--out", OUT, "--work-dir", WORK, "--data-dir", DATA,
        check=False)
print("validation exit code:", rc, "(0 = no structural violation)")
sh(sys.executable, "-m", "src.m2.stats", "--out", OUT, "--work-dir", WORK)

In [ ]:
# ---- Measurements ----------------------------------------------------------------------------
runs = sorted((WORK / "runs").glob("*.json"))
last = json.load(open(runs[-1], encoding="utf-8"))
keys = ["papers", "workers", "done", "skipped", "failed", "timeouts", "wall_s", "paper_ms_mean",
        "paper_ms_median", "paper_ms_p95", "version_ms_mean", "version_ms_median", "version_ms_p95",
        "peak_rss_mb", "peak_workers_rss_mb", "paragraph_cache_hit_rate", "reused_versions",
        "output_bytes_total", "work_bytes_total", "status_counts"]
for k in keys:
    print(f"{k:28s} {last.get(k)}")
print("timing:", {k: round(v, 1) for k, v in timing.items()})
summary = json.load(open(WORK / "report_exports" / "summary.json", encoding="utf-8"))
val = json.load(open(WORK / "report_exports" / "validation.json", encoding="utf-8"))
print("success rate (non-empty):", summary.get("success_rate_non_empty"))
print("dedup:", summary.get("dedup"))
for k in ("violations_total", "residual_latex_rate", "cite_coverage", "cite_coverage_papers_with_bib_source",
          "abstract_fidelity_median", "title_fidelity_median"):
    print(f"{k:38s} {val.get(k)}")
bench = dict(last, timing=timing, colab=IN_COLAB, cpus=os.cpu_count())
json.dump(bench, open(WORK / "report_exports" / "benchmark.json", "w", encoding="utf-8"), indent=1)

In [ ]:
# ---- Copy results back to Drive -------------------------------------------------------------
if IN_COLAB and COPY_BACK:
    dst = Path(DRIVE_OUT)
    dst.mkdir(parents=True, exist_ok=True)
    for src, name in ((OUT, "tables"), (WORK, "m2")):
        shutil.copytree(src, dst / name, dirs_exist_ok=True)
    print("copied to", dst)